# 안전모 · 안전조끼 탐지 파인튜닝 — yolov8n (착용만 학습)

관제 서버의 **안전모 미착용(NO_HELMET) · 안전조끼 미착용(NO_VEST)** 판정(`backend/app/ppe.py`)에 쓰는 모델을 만든다.

**착용한 것만 학습한다**: 클래스는 `person` · `helmet` · `vest` 세 개. 미착용(`no-hardhat` 등) 라벨은 버린다.
서버가 작업자 박스의 머리 · 몸통에 안전모 · 조끼 박스가 **없으면** 미착용으로 보고, 추적 번호마다 몇 초 지켜본 뒤 다수결로 정한다.

| 순서 | 내용 |
|---|---|
| 1 | GPU 확인 · 패키지 설치 · 구글 드라이브 연결 |
| 2 | 설정 (데이터 zip 목록 · 클래스 이름 맞추기) |
| 3 | (선택) 우리 현장 영상에서 라벨링할 프레임 뽑기 |
| 4 | 데이터 풀기 · 클래스 확인 |
| 5 | 라벨 합치기 (`person · helmet · vest` 로 바꾸고, 사람 라벨이 없는 데이터는 원래 모델로 채움) |
| 6 | 학습 (결과는 드라이브에 저장, 끊기면 이어서) |
| 7 | 검증 · 예측 확인 |
| 8 | OpenVINO 로 변환해 드라이브에 저장 → 프로젝트에 넣기 |

메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 고른다.

## 1. 준비

In [ ]:
!nvidia-smi -L
%pip install -q "ultralytics>=8.3" openvino
from google.colab import drive
drive.mount("/content/drive")

### 설정
- `DATA_ZIPS`: 데이터 zip 목록 (드라이브 경로). YOLO 형식 + `data.yaml` 이면 두 가지 폴더 구조를 다 받는다:
  - Roboflow 식 `train/images` · `train/labels` · `valid/…` / Ultralytics 식 `images/train` · `labels/train` · `images/val` …
  - `test` 는 검증에 합치고, 검증 폴더가 없으면 train 의 10% 를 떼어 쓴다.
  - 공개 데이터: Ultralytics *Construction-PPE* (AGPL-3.0), Roboflow Universe 의 *Construction Site Safety* · *Hard Hat Workers*, SHWD 등. **라이선스를 꼭 확인한다.**
  - **우리 현장 프레임을 꼭 섞는다** (3단계로 뽑아 Roboflow 에서 라벨링 → YOLOv8 zip). 카메라 각도 · 거리 · 조끼 색이 달라 공개 데이터만으로는 성능이 꽤 떨어진다.
- `CLASS_MAP`: 원래 데이터의 클래스 이름(소문자) → 새 클래스. **여기에 없는 이름은 버린다** (`no-hardhat`, `mask`, `cone` …).
  4단계에서 데이터마다 어떤 이름이 남고 버려지는지 표로 보여 주니, 빠진 이름이 있으면 여기에 더하고 다시 돌린다.
- `IMGSZ` 는 서버의 `YOLO_IMGSZ`(기본 640)와 같아야 한다 (OpenVINO 로 바꿀 때 크기가 고정됨).

In [ ]:
from pathlib import Path

DATA_ZIPS = [
    "/content/drive/MyDrive/construction-ppe.zip",      # ← 공개 PPE 데이터 (Ultralytics Construction-PPE)
    # "/content/drive/MyDrive/datasets/ppe_site.zip",   # ← 우리 현장 프레임에 라벨링한 것
]
OUT_DIR = "/content/drive/MyDrive/yolo-runs"  # 학습 결과 저장 (드라이브라 런타임이 끊겨도 남음)

NAMES = ["person", "helmet", "vest"]
CLASS_MAP = {
    "person": "person", "worker": "person", "people": "person",
    "hardhat": "helmet", "hard-hat": "helmet", "hard hat": "helmet", "helmet": "helmet", "safety helmet": "helmet",
    "safety vest": "vest", "safety-vest": "vest", "safety_vest": "vest", "vest": "vest", "reflective vest": "vest",
}
PSEUDO_CONF = 0.4   # 사람 라벨이 없는 데이터는 원래 yolov8n(COCO)이 이 confidence 이상으로 잡은 사람을 채운다
VALID_SHARE = 0.1   # valid/ 가 없는 데이터는 train 에서 이만큼 떼어 검증에 씀

EPOCHS, IMGSZ, BATCH = 100, 640, 16

SRC = Path("/content/src")   # 받은 원본 (zip 마다 폴더 하나)
DST = Path("/content/data")  # 합친 학습 데이터

## 3. (선택) 우리 현장 영상에서 라벨링할 프레임 뽑기
`storage/videos/` 의 영상을 드라이브 `VIDEO_DIR` 에 올리고 돌리면, `EVERY_SEC` 초마다 한 장씩 뽑아 zip 으로 드라이브에 저장한다.
이 zip 을 Roboflow 에 올려 `person` · `helmet` · `vest` 를 라벨링하고 **YOLOv8** 형식으로 내보내 `DATA_ZIPS` 에 더한다.
작업자가 나오는 장면 위주로 수백 장이면 충분히 효과가 있다. 안전모 · 조끼를 **안 한** 사람도 꼭 섞는다 (그 사람은 `person` 만 라벨).

In [ ]:
import cv2, shutil

VIDEO_DIR = Path("/content/drive/MyDrive/datasets/site_videos")  # ← 현장 영상 폴더
EVERY_SEC = 2
FRAMES = Path("/content/site_frames")

if VIDEO_DIR.exists():
    shutil.rmtree(FRAMES, ignore_errors=True); FRAMES.mkdir(parents=True)
    for video in sorted(VIDEO_DIR.iterdir()):
        cap = cv2.VideoCapture(str(video))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30
        step, n, saved = max(1, round(fps * EVERY_SEC)), 0, 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if n % step == 0:
                cv2.imwrite(str(FRAMES / f"{video.stem}_{n:06d}.jpg"), frame); saved += 1
            n += 1
        print(f"{video.name}: {saved}장")
    zip_path = shutil.make_archive(str(VIDEO_DIR.parent / "site_frames"), "zip", FRAMES)
    print("라벨링할 프레임:", zip_path)
else:
    print("VIDEO_DIR 이 없어 건너뜀:", VIDEO_DIR)

## 4. 데이터 풀기 · 클래스 확인
zip 마다 `data.yaml` 의 클래스 이름을 읽어 `CLASS_MAP` 으로 바꾼다. **남는 것 · 버리는 것**을 꼭 눈으로 확인한다.

In [ ]:
import random, shutil, zipfile, yaml

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def split_dirs(root, s):
    # (사진 폴더, 라벨 폴더). Roboflow 식 train/images 와 Ultralytics 식 images/train 둘 다 받는다
    for img, lbl in ((root / s / "images", root / s / "labels"), (root / "images" / s, root / "labels" / s)):
        if img.exists():
            return img, lbl
    return None

shutil.rmtree(SRC, ignore_errors=True)
sources = []  # [(이름, 폴더, {원래 번호: 새 번호}, [split])]
for k, zp in enumerate(DATA_ZIPS):
    root = SRC / f"src{k}"
    with zipfile.ZipFile(zp) as z:
        z.extractall(root)
    inner = [p for p in root.iterdir() if p.is_dir()]
    if not (root / "data.yaml").exists() and len(inner) == 1:  # zip 안에 폴더가 한 겹 더 있으면
        root = inner[0]
    names = yaml.safe_load((root / "data.yaml").read_text())["names"]
    names = names if isinstance(names, dict) else dict(enumerate(names))
    remap = {int(i): NAMES.index(CLASS_MAP[n.strip().lower()]) for i, n in names.items() if n.strip().lower() in CLASS_MAP}
    splits = [s for s in ("train", "valid", "val", "test") if split_dirs(root, s)]
    assert "train" in splits, f"{zp}: train 사진 폴더가 있어야 합니다"
    sources.append((Path(zp).stem, root, remap, splits))
    print(f"■ {Path(zp).name}  ({' · '.join(splits)})")
    for i, n in names.items():
        print(f"   {int(i):2d} {n:24s} → {NAMES[remap[int(i)]] if int(i) in remap else '(버림)'}")
    if NAMES.index("person") not in remap.values():
        print("   ※ 사람 라벨이 없음 → 5단계에서 원래 yolov8n 으로 채운다")

## 5. 라벨 합치기
- 원래 번호 → 새 번호로 바꾸고, `CLASS_MAP` 에 없는 클래스 줄은 버린다. 다각형(세그) 라벨은 박스로 바꾼다.
- 사람 라벨이 없는 데이터는 원래 yolov8n(COCO)으로 사람을 찾아 채운다 (pseudo-label). 사람이 없으면 서버가 안전모 · 조끼를 짝지을 수 없다.
- 이름이 겹치지 않게 사진 파일 이름 앞에 데이터 이름을 붙인다.

In [ ]:
from tqdm.auto import tqdm
from ultralytics import YOLO

coco = YOLO("yolov8n.pt")
PERSON = NAMES.index("person")

def to_box(nums):
    """박스(cx cy w h) 는 그대로, 다각형(x y x y …)은 감싸는 박스로"""
    if len(nums) == 4:
        return nums
    xs, ys = nums[0::2], nums[1::2]
    x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
    return [(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1]

def line(cls, box):
    return f"{cls} " + " ".join(f"{min(max(v, 0), 1):.6f}" for v in box)

shutil.rmtree(DST, ignore_errors=True)
for s in ("train", "valid"):
    (DST / s / "images").mkdir(parents=True); (DST / s / "labels").mkdir(parents=True)
count = {n: 0 for n in NAMES}
random.seed(0)
for name, root, remap, splits in sources:
    need_person = PERSON not in remap.values()
    has_valid = "valid" in splits or "val" in splits
    for s in splits:
        img_dir, lbl_dir = split_dirs(root, s)
        imgs = sorted(p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXT)
        for img in tqdm(imgs, desc=f"{name}/{s}"):
            # 검증 폴더가 없는 데이터는 train 에서 VALID_SHARE 만큼 떼어 검증에, test 는 검증에 합친다
            dst = "valid" if s != "train" or (not has_valid and random.random() < VALID_SHARE) else "train"
            rows = []
            lbl = lbl_dir / (img.stem + ".txt")
            for raw in (lbl.read_text().splitlines() if lbl.exists() else []):
                v = raw.split()
                if len(v) < 5 or int(v[0]) not in remap:
                    continue
                cls = remap[int(v[0])]
                rows.append(line(cls, to_box(list(map(float, v[1:])))))
                count[NAMES[cls]] += 1
            if need_person:
                r = coco(str(img), classes=[0], conf=PSEUDO_CONF, verbose=False)[0]
                for box in r.boxes.xywhn.tolist():
                    rows.append(line(PERSON, box)); count["person"] += 1
            out = f"{name}_{img.stem}"
            shutil.copy(img, DST / dst / "images" / (out + img.suffix.lower()))
            (DST / dst / "labels" / (out + ".txt")).write_text("\n".join(rows))

data_yaml = DST / "data.yaml"
data_yaml.write_text(f"path: {DST}\ntrain: train/images\nval: valid/images\n"
                     + "names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(NAMES)))
for s in ("train", "valid"):
    print(s, len(list((DST / s / "images").iterdir())), "장")
print("라벨 수:", count)

### 라벨 눈으로 확인 (몇 장)
사람(초록)에 안전모(노랑) · 조끼(주황)가 제자리에 붙어 있는지 본다. 이상하면 `CLASS_MAP` 을 고치고 4 · 5단계를 다시 돌린다.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

COLORS = ["#3ddc84", "#ffd60a", "#ff9a1f"]
pool = sorted((DST / "train" / "images").iterdir())
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, img in zip(axes.flat, random.sample(pool, k=min(6, len(pool)))):
    im = Image.open(img).convert("RGB"); W, H = im.size
    ax.imshow(im); ax.set_title(img.name, fontsize=9); ax.axis("off")
    for row in (DST / "train" / "labels" / (img.stem + ".txt")).read_text().splitlines():
        c, cx, cy, w, h = row.split(); c = int(c)
        cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
        ax.add_patch(patches.Rectangle((cx - w / 2, cy - h / 2), w, h, fill=False, lw=2, ec=COLORS[c]))
plt.tight_layout(); plt.show()
print("초록 = 사람 · 노랑 = 안전모 · 주황 = 조끼")

## 6. 학습
COCO 로 학습된 `yolov8n.pt` 에서 시작한다. 결과는 `OUT_DIR/ppe` 에 쌓인다.
런타임이 끊기면 1 · 설정 셀만 다시 돌리고 **이어서 학습** 셀을 실행한다.

In [ ]:
model = YOLO("yolov8n.pt")
model.train(data=str(data_yaml), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=20,
            project=OUT_DIR, name="ppe", exist_ok=True, close_mosaic=10, plots=True)

In [ ]:
# 이어서 학습 (끊겼을 때만)
YOLO(f"{OUT_DIR}/ppe/weights/last.pt").train(resume=True)

## 7. 검증 · 예측 확인
클래스별 mAP 와 **recall** 을 본다. 서버는 안전모 · 조끼를 *못 찾으면 미착용*으로 보므로, 이 모델은 precision 보다 **recall 이 중요**하다.
안전모 recall 이 낮으면 미착용 오탐이 늘어난다 → 작게 보이는 안전모 사진 · 우리 현장 프레임을 더 넣는다.

In [ ]:
BEST = f"{OUT_DIR}/ppe/weights/best.pt"
best = YOLO(BEST)
m = best.val(data=str(data_yaml), imgsz=IMGSZ, split="val")
for j, c in enumerate(m.box.ap_class_index):  # 검증 데이터에 나온 클래스 순서
    p, r, ap50, ap = m.box.class_result(j)
    print(f"{NAMES[c]:7s} precision {p:.3f} · recall {r:.3f} · mAP50 {ap50:.3f} · mAP50-95 {ap:.3f}")

In [ ]:
from IPython.display import display
for img in random.sample(sorted((DST / "valid" / "images").iterdir()), k=3):
    r = best(str(img), imgsz=IMGSZ, conf=0.3, verbose=False)[0]
    display(Image.fromarray(r.plot()[:, :, ::-1]))

## 8. OpenVINO 로 변환 → 프로젝트에 넣기

In [ ]:
ov_dir = best.export(format="openvino", imgsz=IMGSZ)
shutil.make_archive(f"{OUT_DIR}/ppe_openvino_model", "zip", ov_dir)
shutil.copy(BEST, f"{OUT_DIR}/ppe.pt")
print("드라이브에 저장:", f"{OUT_DIR}/ppe.pt", "·", f"{OUT_DIR}/ppe_openvino_model.zip")

### 프로젝트에 연결하는 법
1. 드라이브의 `ppe.pt` 를 `vision-safety-control/backend/` 에 넣는다.
   서버가 처음 켜질 때 같은 이름의 `.pt` 를 찾아 `ppe_openvino_model/` 로 바꿔 둔다 (zip 을 풀어 그 폴더로 넣어도 된다).
2. `backend/.env` 에 `YOLO_PPE_MODEL=ppe_openvino_model` 을 넣고 서버를 다시 켠다.
   - 기본 모델(`YOLO_MODEL`)은 그대로 둔다. 사람 · 트럭은 기본 모델 것을 쓰고, 이 모델에서는 **안전모 · 조끼 박스만** 더한다 (장비 근접 판정이 그대로 돌게).
   - 모델을 하나 더 돌리는 만큼 탐지가 느려진다.
3. 판정 기준 화면(8-8)의 안전모 미착용 · 안전조끼 미착용에 `YOLO 판정` 표시가 붙으면 연결된 것이다.
   - 작업자 confidence 기준 · 쿨다운 · 사용 여부는 그 화면에서 바꾼다.
4. 오탐이 많으면 `.env` 로 조절한다:
   - `PPE_PART_CONF` (기본 0.30): 안전모 · 조끼 박스를 착용 증거로 받는 confidence. 낮출수록 미착용 오탐이 준다.
   - `PPE_WINDOW_SEC` (기본 3): 작업자마다 지켜보는 시간. 늘리면 잠깐 가려진 경우에 덜 속는다.
   - `PPE_MIN_PERSON_PX` (기본 60): 이보다 작게 보이는 작업자는 판정하지 않는다 (머리가 너무 작음).